In [1]:
# Load model directly
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained("../pre_trained_models/ashaar")
model = AutoModelForCausalLM.from_pretrained("../pre_trained_models/ashaar")

/home/hussam/python_envs/complete_venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [13]:
# text_input = "مرحبا"

In [14]:
# # original one
# encoded_input = tokenizer(text_input, return_tensors='pt')
# output = model.generate(**encoded_input, max_length = 512, top_p = 0.95, do_sample=True)

In [15]:
# output_sequences = model.generate(
#     input_ids=encoded_input["input_ids"],
#     max_new_tokens=50,
#     temperature=0.7, # Controls randomness (lower = more deterministic)
#     do_sample=True, # Enables sampling
#     pad_token_id=tokenizer.eos_token_id # Prevents warnings about pad token
# )

In [16]:
# generated_text = tokenizer.decode(output[0], skip_special_tokens=True)
# print(generated_text)

In [17]:
# for x in range(len(generated_text)-1):
#     if generated_text[x] == " ":
#         if generated_text[x+1] != " ":
#             continue
#     print(generated_text[x],end="")

In [2]:
import json 
theme_to_token = json.load(open("../pre_trained_models/ashaar/tokens/theme_tokens.json", "r"))
token_to_theme = {t:m for m,t in theme_to_token.items()}

meter_to_token = json.load(open("../pre_trained_models/ashaar/tokens/meter_tokens.json", "r"))
token_to_meter = {t:m for m,t in meter_to_token.items()}

In [37]:
theme = "قصيدة غزل"
meter = "الطويل"
qafiyah = "ر"
prompt = f"{meter_to_token[meter]} {qafiyah} {theme_to_token[theme]}"
print(prompt)
encoded_input = tokenizer(prompt, return_tensors='pt')
output = model.generate(**encoded_input, max_length = 512, top_p = 3, do_sample=True)
print(encoded_input)

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


<|meter_13|> ر <|theme_8|>
{'input_ids': tensor([[20, 93, 53, 32]]), 'attention_mask': tensor([[1, 1, 1, 1]])}


In [38]:
result = ""
prev_token = ""

for i, beam in enumerate(output[:, len(encoded_input.input_ids[0]):]):
    for token in beam:
        # print(decoded)
        decoded = tokenizer.decode(token)
        if 'meter' in decoded or 'theme' in decoded:
            break
        if decoded == "<|vsep|>":
            result += "\n\n"
        elif decoded in ["<|bsep|>", "</|bsep|>"]:
            result += "\n"
        elif decoded in ['<|psep|>', '</|psep|>']:
            pass
        else:
            result += decoded
        prev_token = decoded
    else:
        break
print(theme+" "+ f"من بحر {meter} مع قافية بحر ({qafiyah})" + "\n" +result)

قصيدة غزل من بحر الطويل مع قافية بحر (ر)

أَلا حَبَّذا غُرُّ الصِبا وَالبَراهِمِ 

مِن لَيلَةٍ فيها تَروحُ وَتَبكُرُ 

وَتَبدو لِعَيني هاشِمِيّ كَأَنَّما 

رَعَت دينَ أَبراهيمَ فيهِ المُعَوَّرُ 

قَضى وَطراً مِن دَنأَةِ النَفسِ وَالهَوى 

قَضى وَطَراً مِنهُ وَجارٍ وَمُجوِرُ 

قَضى وَطَراً مِن ماءِ دُنياهُ مَعرِفاً 

وَما لَم يَكُن في غَيرِهِ اليَومَ مُنكَرُ 

وَأَحيا بِهِ رَمساً وَغادَرَ نِسوَةً 

بِها غَدِراتُ السافِلاتِ تُغَدَّرُ 

